# Movie Review Sentiment Classification — Fine-Tuning Distilbert-base-uncased in Colab Notebook

In [40]:
from google.colab import userdata
import os

# Load the secret into Python
my_secret = userdata.get('HF_Token')

# Optional: Set it as a system environment variable
os.environ['HF_TOKEN'] = my_secret


In [42]:
!hf auth whoami

✓ Logged in
  user: A-Asif


In [3]:
!pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


In [5]:
RAW_PATH = "/content/ReviewClassification.csv"
OUT_DIR = "/content/data"

import os
os.makedirs(OUT_DIR, exist_ok=True)

In [6]:
import pandas as pd

df = pd.read_csv(RAW_PATH)

In [7]:
if df.columns[0].startswith("Unnamed"):
  df = df.drop(columns=[df.columns[0]])

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

In [63]:
print(df.isnull().sum())

Review       0
sentiment    0
label        0
dtype: int64


In [64]:
print(f"duplicates: {df.duplicated().sum()}")

duplicates: 0


In [8]:
df = df.drop_duplicates().reset_index(drop=True)

In [62]:
df.head()

,Review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. The filming tec...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,Petter Mattei's Love in the Time of Money is a...,positive,1


In [65]:
print(df["sentiment"].value_counts())
print()
print(df["sentiment"].value_counts(normalize=True).round(3))

sentiment
positive    1005
negative     995
Name: count, dtype: int64

sentiment
positive    0.502
negative    0.498
Name: proportion, dtype: float64


In [66]:
df["word_count"] = df["Review"].str.split().str.len()
print(df["word_count"].describe())
print(f"95th percentile: {df['word_count'].quantile(0.95):.0f} words")
print(f"99th percentile: {df['word_count'].quantile(0.99):.0f} words")

count    2000.000000
mean      221.584000
std       160.711468
min        18.000000
25%       124.000000
50%       169.000000
75%       271.000000
max      1486.000000
Name: word_count, dtype: float64
95th percentile: 550 words
99th percentile: 842 words


In [9]:
label2id = {"negative": 0, "positive": 1}
id2label = {v: k for k, v in label2id.items()}

df["label"] = df["sentiment"].str.lower().map(label2id)
assert df["label"].isnull().sum() == 0, "Found a sentiment value outside {positive, negative}"

print("label2id =", label2id)

label2id = {'negative': 0, 'positive': 1}


In [10]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.111, stratify=train_val_df["label"], random_state=42
    # 0.111 of the remaining 90% ≈ 10% of the original total
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ratio = split["label"].value_counts(normalize=True).round(3).to_dict()
    print(f"{name:5s}: {len(split):6d} rows | class ratio {ratio}")

train:   1600 rows | class ratio {1: 0.502, 0: 0.498}
val  :    200 rows | class ratio {1: 0.5, 0: 0.5}
test :    200 rows | class ratio {1: 0.505, 0: 0.495}


In [11]:
train_df[["Review", "label"]].to_csv(f"{OUT_DIR}/train.csv", index=False)
val_df[["Review", "label"]].to_csv(f"{OUT_DIR}/val.csv", index=False)
test_df[["Review", "label"]].to_csv(f"{OUT_DIR}/test.csv", index=False)

print(f"Saved train/val/test CSVs to {OUT_DIR}/")

Saved train/val/test CSVs to /content/data/


In [12]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loaded tokenizer for:", MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loaded tokenizer for: distilbert-base-uncased


In [13]:
sample_text = "This movie was surprisingly great, though a bit slow."
encoded = tokenizer(sample_text)

print("Original text:", sample_text)
print()
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
if "token_type_ids" in encoded:
    print("token_type_ids:", encoded["token_type_ids"])

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print()
print("Tokens (▁ marks the start of a new word in SentencePiece):")
print(tokens)

Original text: This movie was surprisingly great, though a bit slow.

input_ids: [101, 2023, 3185, 2001, 10889, 2307, 1010, 2295, 1037, 2978, 4030, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Tokens (▁ marks the start of a new word in SentencePiece):
['[CLS]', 'this', 'movie', 'was', 'surprisingly', 'great', ',', 'though', 'a', 'bit', 'slow', '.', '[SEP]']


In [14]:
import pandas as pd

train_df = pd.read_csv(f"{OUT_DIR}/train.csv")

# Sample a subset for speed if the full set is slow; 5000 rows gives a stable percentile estimate
sample_for_length = train_df.sample(min(5000, len(train_df)), random_state=42)

token_lengths = sample_for_length["Review"].apply(
    lambda t: len(tokenizer(t, truncation=False)["input_ids"])
)

print(token_lengths.describe())
p95 = int(token_lengths.quantile(0.95))
p99 = int(token_lengths.quantile(0.99))
print(f"\n95th percentile: {p95} tokens")
print(f"99th percentile: {p99} tokens")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (788 > 512). Running this sequence through the model will result in indexing errors


count    1600.000000
mean      268.040000
std       192.098626
min        23.000000
25%       151.000000
50%       204.500000
75%       328.250000
max      1730.000000
Name: Review, dtype: float64

95th percentile: 650 tokens
99th percentile: 1012 tokens


In [15]:
# Inspect the longest reviews for artifacts before trusting the length numbers
longest = sample_for_length.loc[token_lengths.sort_values(ascending=False).index[:3], "Review"]
for i, text in enumerate(longest):
    print(f"--- Review {i} (first 500 chars) ---")
    print(text[:500])
    print()

# Specifically check how common HTML break tags are across the whole training set
has_br_tag = train_df["Review"].str.contains(r"<br\s*/?>", regex=True).mean() * 100
print(f"% of reviews containing '<br />' tags: {has_br_tag:.1f}%")
print(f"\n\nLength of longest text: {len(str(longest))}")

--- Review 0 (first 500 chars) ---
Warning Does contain spoilers.Open Your EyesIf you have not seen this film and plan on doing so just stop reading here and take my word for it. You have to see this film. I have seen it four times so far and I still haven't made up my mind as to what exactly happened in the film. That is all I am going to say because if you have not seen this film then stop reading right now.If you are still reading then I am going to pose some questions to you and maybe if anyone has any answers you can email m

--- Review 1 (first 500 chars) ---
I thought that ROTJ was clearly the best out of the three Star Wars movies. I find it surprising that ROTJ is considered the weakest installment in the Trilogy by many who have voted. To me it seemed like ROTJ was the best because it had the most profound plot the most suspense surprises most emotionalespecially the ending and definitely the most episodic movie. I personally like the Empire Strikes Back a lot also but I thin

In [16]:
# DeBERTa-v3-large's hard limit is 512 tokens (position embeddings stop there).
# Pick the smallest reasonable length that covers most of the distribution.
if p95 <= 256:
    chosen_max_length = 256
elif p95 <= 384:
    chosen_max_length = 384
else:
    chosen_max_length = 512

pct_truncated = (token_lengths > chosen_max_length).mean() * 100
print(f"Chosen max_length = {chosen_max_length}")
print(f"This truncates ~{pct_truncated:.1f}% of sampled training reviews.")

Chosen max_length = 512
This truncates ~9.4% of sampled training reviews.


In [17]:
def tokenize_batch(batch):
    return tokenizer(
        batch["Review"],
        truncation=True,
        max_length=chosen_max_length,
        padding=False,
    )

# Smoke test on a few rows
test_batch = {"Review": train_df["Review"].head(3).tolist()}
result = tokenize_batch(test_batch)
for i in range(3):
    print(f"Row {i}: {len(result['input_ids'][i])} tokens")

Row 0: 512 tokens
Row 1: 315 tokens
Row 2: 57 tokens


In [18]:
from datasets import Dataset, DatasetDict

val_df = pd.read_csv(f"{OUT_DIR}/val.csv")
test_df = pd.read_csv(f"{OUT_DIR}/test.csv")

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)
tokenized_datasets

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['Review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['Review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
})

In [19]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
print(model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "negative",
    "1": "positive"
  },
  "initializer_range": 0.02,
  "label2id": {
    "negative": 0,
    "positive": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



In [21]:
tokenized_datasets = tokenized_datasets.remove_columns(["Review"])

In [22]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [23]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

In [24]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],

        "precision": precision.compute(
            predictions=predictions,
            references=labels
        )["precision"],

        "recall": recall.compute(
            predictions=predictions,
            references=labels
        )["recall"],

        "f1": f1.compute(
            predictions=predictions,
            references=labels
        )["f1"],
    }

In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./deberta_movie_review2",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=50,

    learning_rate=2e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=4,

    num_train_epochs=10,

    weight_decay=0.01,

    warmup_ratio=0.1,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),

    report_to="none",

    save_total_limit=2
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
!pip install --upgrade torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.131105,0.192024,0.920000,0.903846,0.940000,0.921569
2,0.717873,0.248569,0.935000,0.914286,0.960000,0.936585
3,0.458315,0.321808,0.915000,0.936842,0.890000,0.912821
4,0.361501,0.487432,0.905000,0.965517,0.840000,0.898396
5,0.071359,0.389975,0.920000,0.946809,0.890000,0.917526
6,0.008799,0.421127,0.940000,0.931373,0.950000,0.940594
7,0.008652,0.460793,0.935000,0.914286,0.960000,0.936585
8,0.000748,0.490682,0.935000,0.930693,0.940000,0.935323
9,0.049704,0.485252,0.930000,0.930000,0.930000,0.930000
10,0.000639,0.490846,0.935000,0.930693,0.940000,0.935323


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.27809374260902403, metrics={'train_runtime': 962.1771, 'train_samples_per_second': 16.629, 'train_steps_per_second': 1.039, 'total_flos': 1639720786822944.0, 'train_loss': 0.27809374260902403, 'epoch': 10.0})

# ***Uploading Finetuned model to HF Repo for future use***

In [36]:
from huggingface_hub import notebook_login

notebook_login()

In [53]:
SAVE_DIR = "./final_model"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [54]:
!ls -lh ./final_model

total 257M
-rw-r--r-- 1 root root  784 Aug  1 05:50 config.json
-rw------- 1 root root 256M Aug  1 05:50 model.safetensors
-rw-r--r-- 1 root root  351 Aug  1 05:50 tokenizer_config.json
-rw-r--r-- 1 root root 695K Aug  1 05:50 tokenizer.json
-rw-r--r-- 1 root root 5.1K Aug  1 05:50 training_args.bin


In [55]:
from huggingface_hub import HfApi

api = HfApi()

In [56]:
api.upload_folder(
    folder_path="./final_model",
    repo_id="A-Asif/movie-review-sentiment-distilbert-Ft",
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/A-Asif/movie-review-sentiment-distilbert-FT/commit/92ca545984db15f9c2fdb18efd96ece55658c5d8', commit_message='Upload folder using huggingface_hub', commit_description='', oid='92ca545984db15f9c2fdb18efd96ece55658c5d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/A-Asif/movie-review-sentiment-distilbert-FT', endpoint='https://huggingface.co', repo_type='model', repo_id='A-Asif/movie-review-sentiment-distilbert-FT'), pr_revision=None, pr_num=None)

# ***Testing uploaded model***

In [57]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "A-Asif/movie-review-sentiment-distilbert-FT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("✅ Model loaded successfully!")

config.json:   0%|          | 0.00/784 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded successfully!


In [59]:
import torch

review = "I will not recommend this movie to anyone, as its graphics were very bad"

inputs = tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print(model.config.id2label[prediction])

negative


In [60]:
import torch

review = "This movie was absolutely amazing. I loved every minute of it."

inputs = tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

print(model.config.id2label[prediction])

positive
